In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from HelperFuncs import build_data, PredictiveChanger

# Predictive Coding Run

In [ ]:
#We use the previous instantiated model so that we can compare predictive coding with standard back-prop from the same initialization
#This makes the comparison as close to apples-apples as possible
#If you are running this for the first time you must first run CreateStart.py which creates and saves the initialized model at models/initial.pt
pred_model = torch.load('models/initial.pt',weights_only=False)

In [4]:
#We need one optimizer for each layer since each layer is optimized only locally rather than globally
optimizers = ([torch.optim.AdamW(pred_model.layer1.parameters(),lr=2e-3)]+
              [torch.optim.AdamW(layer.parameters(),2e-3) for layer in pred_model.layers]+
              [torch.optim.AdamW(pred_model.final.parameters(),lr=2e-3)])
objective = nn.MSELoss()

In [6]:
x_test, y_test = build_data(200)

In [7]:
n_epochs = 1000
for epoch in range(n_epochs):
    x,y = build_data(100)
    final_preds, intermediate_preds = pred_model(x)
    pred_model.predictive_optimize(x,intermediate_preds,y,optimizers,objective)
    with torch.no_grad():
        test_preds, test_intermediate_preds = pred_model(x_test)
        test_loss = objective(test_preds.ravel(), y_test)
        if test_loss<10:
            optimizers = ([torch.optim.AdamW(pred_model.layer1.parameters(),lr=1e-6)]+
                          [torch.optim.AdamW(layer.parameters(),1e-6) for layer in pred_model.layers]+
                          [torch.optim.AdamW(pred_model.final.parameters(),lr=1e-6)])
        elif test_loss<20:
            optimizers = ([torch.optim.AdamW(pred_model.layer1.parameters(),lr=2e-6)]+
                          [torch.optim.AdamW(layer.parameters(),2e-6) for layer in pred_model.layers]+
                          [torch.optim.AdamW(pred_model.final.parameters(),lr=2e-6)])
        elif test_loss<40:
            optimizers = ([torch.optim.AdamW(pred_model.layer1.parameters(),lr=1e-5)]+
                          [torch.optim.AdamW(layer.parameters(),1e-5) for layer in pred_model.layers]+
                          [torch.optim.AdamW(pred_model.final.parameters(),lr=1e-5)])
        elif test_loss<100:
            optimizers = ([torch.optim.AdamW(pred_model.layer1.parameters(),lr=2e-5)]+
                          [torch.optim.AdamW(layer.parameters(),2e-5) for layer in pred_model.layers]+
                          [torch.optim.AdamW(pred_model.final.parameters(),lr=2e-5)])
        elif test_loss<200:
            optimizers = ([torch.optim.AdamW(pred_model.layer1.parameters(),lr=1e-4)]+
                          [torch.optim.AdamW(layer.parameters(),1e-4) for layer in pred_model.layers]+
                          [torch.optim.AdamW(pred_model.final.parameters(),lr=1e-4)])
        elif test_loss<400:
            optimizers = ([torch.optim.AdamW(pred_model.layer1.parameters(),lr=1e-3)]+
                          [torch.optim.AdamW(layer.parameters(),1e-3) for layer in pred_model.layers]+
                          [torch.optim.AdamW(pred_model.final.parameters(),lr=1e-3)])
    print(epoch,round(test_loss.detach().item(),4))

0 516.7094
1 515.449
2 514.1756
3 512.7826
4 511.25
5 509.541
6 507.5815
7 505.2281
8 502.4269
9 499.0395
10 494.7498
11 489.325
12 482.1615
13 472.6262
14 459.7371
15 442.4329
16 418.5002
17 385.7111
18 366.2075
19 343.3605
20 316.5343
21 285.6172
22 249.9429
23 210.598
24 169.2738
25 165.0078
26 160.7615
27 156.505
28 152.3658
29 148.1892
30 143.9496
31 139.7222
32 135.5184
33 131.2959
34 127.1221
35 122.9359
36 118.7752
37 114.6539
38 110.5647
39 106.5114
40 102.48
41 98.502
42 97.7066
43 96.9103
44 96.131
45 95.3545
46 94.5791
47 93.8023
48 93.027
49 92.2457
50 91.46
51 90.6807
52 89.9026
53 89.1252
54 88.3549
55 87.5799
56 86.812
57 86.0389
58 85.2674
59 84.4948
60 83.725
61 82.9509
62 82.194
63 81.4225
64 80.6584
65 79.8899
66 79.1385
67 78.3889
68 77.6555
69 76.8936
70 76.1593
71 75.4194
72 74.6699
73 73.9262
74 73.1886
75 72.4411
76 71.7068
77 70.9743
78 70.2434
79 69.512
80 68.7865
81 68.0627
82 67.341
83 66.6263
84 65.9076
85 65.191
86 64.4826
87 63.7705
88 63.0706
89 62.3653

# Standard back-propogation Run

In [ ]:
pred_model = torch.load('models/initial.pt',weights_only=False)

In [9]:
optimizer = torch.optim.AdamW(pred_model.parameters(),lr=2e-3)
objective = nn.MSELoss()

In [ ]:
n_epochs = 1000
for epoch in range(n_epochs):
    x,y = build_data(100)
    final_preds, intermediate_preds = pred_model(x)
    loss = objective(final_preds.ravel(),y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    with torch.no_grad():
        test_preds, test_intermediate_preds = pred_model(x_test)
        test_loss = objective(test_preds.ravel(), y_test)
        #we use the same learning rate trimming as above to make these two tests as comparable as possible.
        if test_loss<10:
            optimizer = torch.optim.AdamW(pred_model.parameters(),lr=1e-6)
        elif test_loss<20:
            optimizer = torch.optim.AdamW(pred_model.parameters(),lr=2e-6)
        elif test_loss<40:
            optimizer = torch.optim.AdamW(pred_model.parameters(),lr=1e-5)
        elif test_loss<100:
            optimizer = torch.optim.AdamW(pred_model.parameters(),lr=2e-5)
        elif test_loss<200:
            optimizer = torch.optim.AdamW(pred_model.parameters(),lr=1e-4)
        elif test_loss<400:
            optimizer = torch.optim.AdamW(pred_model.parameters(),lr=1e-3)
    print(epoch,round(test_loss.detach().item(),4))

0 517.4788
1 515.9103
2 514.4274
3 513.0193
4 511.4782
5 509.6792
6 507.6292
7 504.9758
8 501.2373
9 495.683
10 486.8691
11 471.9269
12 444.387
13 389.8235
14 322.1294
15 207.0815
16 49.1235
17 46.3382
18 43.6151
19 40.9619
20 38.3818
21 37.1235
22 35.885
23 34.6695
24 33.4737
25 32.3015
26 31.1555
27 30.0311
28 28.9323
29 27.8586
30 26.8135
31 25.7949
32 24.8049
33 23.8427
34 22.9102
35 22.0112
36 21.1423
37 20.3073
38 19.5051
39 19.3492
40 19.1944
41 19.0413
42 18.8896
43 18.7392
44 18.5904
45 18.4429
46 18.2969
47 18.1524
48 18.0094
49 17.8678
50 17.7278
51 17.5894
52 17.4524
53 17.3172
54 17.1833
55 17.051
56 16.92
57 16.7907
58 16.6627
59 16.5366
60 16.4119
61 16.2889
62 16.1673
63 16.0477
64 15.9295
65 15.8128
66 15.6979
67 15.5845
68 15.4729
69 15.3628
70 15.2544
71 15.1478
72 15.0428
73 14.9397
74 14.8379
75 14.7379
76 14.6396
77 14.543
78 14.448
79 14.3552
80 14.2638
81 14.1744
82 14.0867
83 14.0008
84 13.9167
85 13.8348
86 13.7544
87 13.6754
88 13.5984
89 13.5233
90 13.4503
9

# Remarks

In this example, predictive coding achieved a better test loss, but repeats of this experiment with other initializations reveal that this is not always the case. At best, the results are inconclusive as far as which method gives a better optimization. It's also worth noting that this was done with completely synthetic data, applying this to a real world data set may be worthwhile to see if one or the other approach handles the inherent messiness of real world data better. Last of all, I'd like to note that the standard back-propogation runs nearly 5x faster.